# 03 · Hysteresis and lock-in — variational costs of changing your mind

A focused experiment on the paper's central claim (§3.2): the complexity
term in the variational free energy is most naturally read as a *motivational
cost of mind-change*, not a generic regulariser. The cost is parameterised
by $\lambda \geq 0$ in the asymmetric-KL update

$$
C^{(t+1)} \;=\; \arg\min_{C} \left[\, \mathrm{KL}\!\bigl(q^{(t+1)} \,\Vert\, p_C\bigr) \;+\; \lambda \cdot \mathrm{KL}\!\bigl(p_C \,\Vert\, p_{C^{(t)}}\bigr)\,\right].
$$

At $\lambda = 0$ the agent freely projects every new posterior onto $C$ —
Bayes-optimal, no resistance. As $\lambda \to \infty$, $C$ is frozen.
Intermediate $\lambda$ is *stubbornness*: the agent updates but pays a
cost, and that cost shows up as **path-dependence under environmental
reversal** (Proposition 1).

**Operational measure.** When we shock the env back-and-forth on a single
feature, the agent's preference $\sigma(C)$ traces a loop in the
$(s^*, \sigma(C))$ phase plane. The loop's *area* is the variational cost
made visible: $\lambda = 0 \Rightarrow$ loop closes exactly $\Rightarrow$
area = 0; larger $\lambda$ $\Rightarrow$ wider loop. Prop 1 says area
scales monotonically with $\lambda$.

This notebook walks through the claim in five steps:
- **§2** — trajectories of an *isolated mind* (no social channel) at four
  $\lambda$ values under a forward-reverse env shock.
- **§3** — phase-space loops at each $\lambda$ — the visual signature.
- **§4** — sweep $\lambda$ over an order-of-magnitude range; loop area
  vs $\lambda$ on a log axis (Prop 1, numerically).
- **§5** — repeated cycles — does the residual drift compound?
- **§6** — turn the social channel back on (vary delegation $d$): how does
  group context modify the individual variational cost?

## §0 · Protocol and setup

We use a single-feature world ($R = 1$, so every step is informative about
the same belief) and drive `env.s_star[0]` through a deterministic pattern:

  warmup at 0 (let beliefs settle) $\to$ shock to 1 (forward) $\to$ shock
  back to 0 (reverse) $\to$ optionally repeat.

For §2–§5 we **disable the social channel** by setting each agent's
delegation factor $d_i = 0$ via an `equinox.tree_at` override. Then
$q_i = p_\mathrm{env}$ exactly and the resulting $C_i$ trajectory is the
pure individual cost-of-mind-change dynamic. §6 turns $d$ back on.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import jax.numpy as jnp
import equinox as eqx

from src import build, ModelConfig, EnvConfig, EvolutionRegime

sns.set_theme(context='notebook', style='whitegrid')


def shoelace_area(xs: np.ndarray, ys: np.ndarray) -> float:
    """Auto-closed polygon shoelace area (returns absolute value)."""
    xs_c = np.concatenate([xs, xs[:1]])
    ys_c = np.concatenate([ys, ys[:1]])
    return 0.5 * float(np.abs(np.sum(xs_c[:-1] * ys_c[1:] - xs_c[1:] * ys_c[:-1])))


def isolate(pop):
    """Override delegation d to zero so q = p_env exactly (no social channel)."""
    return eqx.tree_at(lambda p: p.d, pop, jnp.zeros_like(pop.d))


def run_shock_cycles(lambda_val: float, *,
                     n_agents: int = 50, warm_T: int = 30, leg_T: int = 20,
                     n_cycles: int = 1, d_mean: float | None = None,
                     gamma_env: float = 4.0, seed: int = 0):
    """Drive a homogeneous-λ population through n_cycles forward-reverse shocks.

    ``d_mean=None``  → disable social channel (isolated mind, d=0).
    ``d_mean=μ``     → sample d ~ Beta(10μ, 10(1-μ)) so mean d ≈ μ.

    Returns ``(sig_traj, x_traj, cycle_idx)`` where
      sig_traj  : (T_snap, n_agents) — σ(C[:, 0]) per step
      x_traj    : (T_snap,)          — s_star[0] per step
      cycle_idx : (T_snap,)          — cycle id (-1 during warmup).
    """
    if d_mean is None:
        d_a, d_b = 0.1, 100.0      # placeholder; overridden by isolate()
    else:
        d_a = max(0.5, 10.0 * d_mean)
        d_b = max(0.5, 10.0 * (1.0 - d_mean))

    cfg = ModelConfig(
        n_agents=n_agents, n_features=1,
        lambda_dist=('constant', {'value': float(lambda_val)}),
        lambda_scope='per_agent',
        gamma_env=gamma_env,
        delegation_alpha=d_a, delegation_beta=d_b,
        # Sparse network: any social coupling is rare; only matters when d>0.
        network_kind='watts_strogatz', mean_degree=4, rewiring_p=0.1,
        seed=seed,
    )
    env_cfg = EnvConfig(
        n_agents=n_agents, n_features=1,
        regime=EvolutionRegime.STATIONARY, stationary_flip_prob=0.0,
        seed=seed + 100,
    )
    env, pop = build(cfg, env_cfg)
    if d_mean is None:
        pop = isolate(pop)

    env.s_star[:] = np.array([0], dtype=np.int64)
    sigs, xs, cycles = [], [], []

    def rec(c: int, x: int) -> None:
        sigs.append(1.0 / (1.0 + np.exp(-np.asarray(pop.C)[:, 0])))
        xs.append(x); cycles.append(c)

    rec(-1, 0)
    for _ in range(warm_T):
        pop, _ = pop.step(env); rec(-1, 0)
    for c in range(n_cycles):
        env.s_star[0] = 1
        for _ in range(leg_T):
            pop, _ = pop.step(env); rec(c, 1)
        env.s_star[0] = 0
        for _ in range(leg_T):
            pop, _ = pop.step(env); rec(c, 0)

    return np.stack(sigs), np.array(xs, dtype=float), np.array(cycles)


# Headline λ grid — spans "freely Bayesian" through "deeply entrenched".
LAMBDAS_HEADLINE = [0.0, 1.0, 5.0, 25.0, 100.0]
LAMBDA_COLOURS = sns.color_palette('viridis', n_colors=len(LAMBDAS_HEADLINE))
print(f'λ headline grid: {LAMBDAS_HEADLINE}')

## §2 · Isolated mind, one shock cycle — trajectories vs $\lambda$

Five agents (one per $\lambda$) all see the same env stimulus:

  warm 30 steps at $s^* = 0$ $\to$ shock to 1 for 20 steps $\to$ shock back to 0 for 20 steps.

With $d = 0$, $q$ is just the env channel $p_\mathrm{env}$ (a sharp Bernoulli
with $\sigma(\gamma_\mathrm{env}) \approx 0.98$ on the observed bit). The
ground truth is `logit(q) ≈ ±4`; whether $C$ tracks it depends entirely
on $\lambda$.

Reading the plot: the dotted black step function is the env truth; each
coloured line is $\sigma(C)$ for one $\lambda$ value. At $\lambda = 0$ the
line snaps onto the env immediately. At $\lambda = 100$ it barely moves.

In [ ]:
warm_T, leg_T = 30, 20
results = {}
for lam in LAMBDAS_HEADLINE:
    sigs, xs, cycles = run_shock_cycles(
        lam, n_agents=8, warm_T=warm_T, leg_T=leg_T, n_cycles=1, d_mean=None,
    )
    results[lam] = (sigs, xs, cycles)

fig, ax = plt.subplots(figsize=(10, 4.5))
_, xs_ref, _ = results[LAMBDAS_HEADLINE[0]]
steps = np.arange(len(xs_ref))
ax.step(steps, xs_ref, where='post', color='black', ls=':', lw=1.0, alpha=0.8,
        label='$s^*$ (env truth)')
for col, lam in zip(LAMBDA_COLOURS, LAMBDAS_HEADLINE):
    sigs, _, _ = results[lam]
    y = sigs.mean(axis=1)              # 8 identical traces (d=0); take mean for safety
    ax.plot(steps, y, color=col, lw=2.0, label=f'\u03bb = {lam:g}')
ax.axvline(warm_T, ls='--', c='grey', lw=0.5)
ax.axvline(warm_T + leg_T, ls='--', c='grey', lw=0.5)
ax.set(xlabel='step', ylabel='$\\sigma(C)$', ylim=(-0.05, 1.05),
       title='Isolated mind: one shock cycle, five variational costs')
ax.legend(loc='center right', fontsize=9)
plt.tight_layout()
plt.show()

## §3 · Phase-space loops at each $\lambda$

Re-plot the same trajectories in the phase plane $(s^*, \sigma(C))$.
Forward leg in orange, reverse leg in blue. A *closed* loop means the
agent returned to its starting equilibrium after the env reversed. An
*open* loop means a residual mind-change cost was unrecovered.

In [ ]:
fig, axes = plt.subplots(1, len(LAMBDAS_HEADLINE),
                          figsize=(3 * len(LAMBDAS_HEADLINE), 3.4),
                          sharey=True)
for ax, lam, col in zip(axes, LAMBDAS_HEADLINE, LAMBDA_COLOURS):
    sigs, xs, cycles = results[lam]
    y = sigs.mean(axis=1)

    # Trajectory by leg — detect leg from x transitions
    post_warm = cycles >= 0
    leg = np.zeros_like(xs, dtype=int)
    in_forward = xs == 1
    leg[in_forward & post_warm] = 1     # 1 = forward leg
    leg[(~in_forward) & post_warm] = 2  # 2 = reverse leg (post-warm with x=0)

    # x jitter so forward (x=1) and reverse (x=0) don't overplot
    ax.plot(xs[leg == 1], y[leg == 1], color='C1', lw=2.0, label='forward')
    ax.plot(xs[leg == 2], y[leg == 2], color='C0', lw=2.0, ls='--', label='reverse')
    ax.scatter([xs[0]], [y[0]], color='black', marker='o', s=70, zorder=5, label='start')
    ax.scatter([xs[-1]], [y[-1]], color='black', marker='s', s=70, zorder=5, label='end')

    area = shoelace_area(xs[post_warm], y[post_warm])
    ax.set(xlabel='$s^*$', xticks=[0, 1], xlim=(-0.15, 1.15), ylim=(-0.05, 1.05),
           title=f'\u03bb = {lam:g}   area = {area:.3f}')
    if lam == LAMBDAS_HEADLINE[0]:
        ax.set_ylabel('$\\sigma(C)$')
        ax.legend(loc='center left', fontsize=8)
fig.suptitle('Hysteresis loops in $(s^*, \\sigma(C))$ — area = unrecovered cost of mind-change',
             y=1.04, fontsize=12)
plt.tight_layout()
plt.show()

## §4 · Loop area vs $\lambda$ — the operational Prop 1 check

Sweep $\lambda$ over more than two orders of magnitude. The naive reading
of Proposition 1 — "loop area is monotone in $\lambda$" — is true *in the
idealised limit* of arbitrarily long shock legs, where every $\lambda$
fully equilibrates and the loop closes only as $\lambda \to 0$. With
**finite** leg duration, the curve is **non-monotone with a peak**:

- $\lambda = 0$: instant snap to truth, loop closes, area = 0.
- Small $\lambda$: partial lag, area grows.
- Intermediate $\lambda$: the agent moves substantially *but incompletely*
  — this is where visible hysteresis is maximised.
- Large $\lambda$: the agent barely moves at all — small $(s^*, \sigma(C))$
  range, small loop area.
- $\lambda \to \infty$: complete stasis, area $\to 0$.

The peak $\lambda^*$ is the **most cognitively biased regime for this
stimulus** — neither flexible enough to track nor rigid enough to ignore.
Its location depends on shock duration, env precision $\gamma_\mathrm{env}$,
and social context (see §6). Prop 1's monotonicity claim describes the
*lower envelope* of this curve as leg length increases.

In [ ]:
lambdas_sweep = np.array([0.0, 0.5, 1.0, 2.0, 5.0, 10.0, 25.0, 50.0, 100.0, 200.0])
areas = np.zeros_like(lambdas_sweep)
for i, lam in enumerate(lambdas_sweep):
    sigs, xs, cycles = run_shock_cycles(
        lam, n_agents=4, warm_T=warm_T, leg_T=leg_T, n_cycles=1, d_mean=None,
    )
    y = sigs.mean(axis=1)
    post_warm = cycles >= 0
    areas[i] = shoelace_area(xs[post_warm], y[post_warm])

peak_i = int(np.argmax(areas))

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(lambdas_sweep, areas, marker='o', lw=2.0, color='C3', markersize=7)
for lam_h, col in zip(LAMBDAS_HEADLINE, LAMBDA_COLOURS):
    idx = np.argmin(np.abs(lambdas_sweep - lam_h))
    ax.scatter([lambdas_sweep[idx]], [areas[idx]], color=col, s=120,
               edgecolor='black', zorder=5, label=f'λ = {lam_h:g} (§2/§3)')
ax.axvline(lambdas_sweep[peak_i], ls='--', c='grey', lw=0.8, alpha=0.7)
ax.annotate(f'peak at λ ≈ {lambdas_sweep[peak_i]:g}',
            xy=(lambdas_sweep[peak_i], areas[peak_i]),
            xytext=(lambdas_sweep[peak_i] * 4, areas[peak_i] - 0.05),
            arrowprops=dict(arrowstyle='->', color='grey'),
            fontsize=10)
ax.set(xscale='symlog', xlabel='variational cost λ (symlog)',
       ylabel='hysteresis loop area',
       title='Loop area vs λ is unimodal — peak at \"stubborn but not frozen\"')
ax.legend(loc='upper right', fontsize=9)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print('λ sweep:')
for lam, area in zip(lambdas_sweep, areas):
    star = '  ← peak' if lam == lambdas_sweep[peak_i] else ''
    print(f'  λ = {lam:>6.1f}  loop area = {area:.4f}{star}')

## §5 · Repeated cycles — does the residual compound?

If the residual cost from one cycle reflects an *equilibrium* lag rather
than *progressive drift*, repeated shocks should produce the same loop
area cycle after cycle. If λ is producing genuine path-dependence, the
loop will instead drift across cycles — each reversal leaves a permanent
residue that the next cycle inherits.

We run 4 cycles per λ and plot loop area per cycle. Stable-area = pure
Prop-1 lag. Growing or shrinking area = state-dependence in the residual.

In [ ]:
n_cycles = 4
per_cycle_area = np.zeros((len(LAMBDAS_HEADLINE), n_cycles))
trajectories = {}
for li, lam in enumerate(LAMBDAS_HEADLINE):
    sigs, xs, cycles = run_shock_cycles(
        lam, n_agents=4, warm_T=warm_T, leg_T=leg_T, n_cycles=n_cycles, d_mean=None,
    )
    y = sigs.mean(axis=1)
    trajectories[lam] = (y, xs, cycles)
    for c in range(n_cycles):
        in_c = cycles == c
        per_cycle_area[li, c] = shoelace_area(xs[in_c], y[in_c])

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# (Left) per-λ trajectory across all cycles
for lam, col in zip(LAMBDAS_HEADLINE, LAMBDA_COLOURS):
    y, xs, _ = trajectories[lam]
    axes[0].plot(np.arange(len(y)), y, color=col, lw=1.8, label=f'\u03bb = {lam:g}')
axes[0].step(np.arange(len(xs)), xs, where='post', color='black', ls=':', lw=1.0,
             alpha=0.7, label='$s^*$')
for cyc in range(n_cycles):
    shock_f = warm_T + cyc * 2 * leg_T + 1
    shock_r = warm_T + (cyc * 2 + 1) * leg_T + 1
    axes[0].axvline(shock_f, ls='--', c='grey', lw=0.3, alpha=0.6)
    axes[0].axvline(shock_r, ls='--', c='grey', lw=0.3, alpha=0.6)
axes[0].set(xlabel='step', ylabel='$\\sigma(C)$',
            title=f'{n_cycles}-cycle trajectories', ylim=(-0.05, 1.05))
axes[0].legend(loc='center right', fontsize=8, ncol=2)

# (Right) per-cycle area, grouped bar
cycle_x = np.arange(1, n_cycles + 1)
bw = 0.15
offsets = np.linspace(-2 * bw, 2 * bw, len(LAMBDAS_HEADLINE))
for li, (lam, col) in enumerate(zip(LAMBDAS_HEADLINE, LAMBDA_COLOURS)):
    axes[1].bar(cycle_x + offsets[li], per_cycle_area[li],
                width=bw, color=col, edgecolor='black', label=f'\u03bb = {lam:g}')
axes[1].set(xlabel='cycle index', ylabel='loop area per cycle', xticks=cycle_x,
            title='Per-cycle loop area — stable = pure lag; growing = drift')
axes[1].legend(loc='upper right', fontsize=8)
plt.tight_layout()
plt.show()

## §6 · The variational cost embedded in a community

Sections 2–5 treated $\lambda$ as a purely individual cognitive cost. When
we let agents listen to each other ($d > 0$), the social channel becomes
an additional source of evidence that gets fed through the same
$\lambda$-gated update — and, critically, the network itself acquires a
*memory* through recursive averaging.

Two predictions, both visible in the heatmap below:

1. **Networks have an emergent variational cost.** At $\lambda = 0$ and
   $d = 0$, the loop area is exactly zero. At $\lambda = 0$ and $d = 0.9$,
   the loop area is large (~0.77) despite every individual being
   Bayes-optimal — because the social channel is a recursive lag (each
   neighbour averages their own neighbours' previous beliefs, which
   averaged *their* previous beliefs...). Group-level inertia is real
   even without individual stubbornness.
2. **Higher $d$ shifts the peak $\lambda^*$ leftward.** When the network
   is doing more of the lock-in work, less individual rigidity is needed
   to maximise visible hysteresis. The cognitive bias regime moves toward
   smaller $\lambda$ as $d$ increases.

The two costs — individual cost-of-mind-change ($\lambda$) and group
cost-of-disagreeing-with-peers ($d$) — substitute for each other in
producing observable hysteresis, with a ridge along $(\lambda + \text{social
inertia}) \approx \text{const}$.

In [ ]:
import time

d_grid = [0.0, 0.3, 0.6, 0.9]
lam_grid = lambdas_sweep                # reuse the sweep grid from §4
area_2d = np.zeros((len(lam_grid), len(d_grid)))

t0 = time.perf_counter()
for li, lam in enumerate(lam_grid):
    for di, dm in enumerate(d_grid):
        sigs, xs, cycles = run_shock_cycles(
            lam, n_agents=80, warm_T=warm_T, leg_T=leg_T, n_cycles=1,
            d_mean=(None if dm == 0.0 else dm),
        )
        y = sigs.mean(axis=1)
        post_warm = cycles >= 0
        area_2d[li, di] = shoelace_area(xs[post_warm], y[post_warm])
print(f'2-D sweep ({len(lam_grid)} λ × {len(d_grid)} d) ran in {time.perf_counter()-t0:.1f}s')

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

# (Left) heatmap
im = axes[0].imshow(area_2d, aspect='auto', origin='lower', cmap='viridis')
axes[0].set(
    xticks=np.arange(len(d_grid)),
    xticklabels=[f'{d:.1f}' for d in d_grid],
    yticks=np.arange(len(lam_grid)),
    yticklabels=[f'{l:g}' for l in lam_grid],
    xlabel='delegation $d$ (social-channel weight)',
    ylabel='variational cost $\\lambda$',
    title='Loop area in $(\\lambda, d)$ space',
)
for li in range(len(lam_grid)):
    for di in range(len(d_grid)):
        axes[0].text(di, li, f'{area_2d[li, di]:.2f}',
                     ha='center', va='center',
                     color='white' if area_2d[li, di] < area_2d.max() / 2 else 'black',
                     fontsize=8)
plt.colorbar(im, ax=axes[0], label='loop area')

# (Right) line plot: area vs λ at each d
d_colours = sns.color_palette('rocket', n_colors=len(d_grid))
for di, (dm, col) in enumerate(zip(d_grid, d_colours)):
    axes[1].plot(lam_grid, area_2d[:, di], marker='o', lw=2.0, color=col,
                 label=f'$d = {dm:.1f}$')
axes[1].set(xscale='symlog', xlabel='$\\lambda$ (symlog)',
            ylabel='loop area',
            title='Loop area vs $\\lambda$ at four conformity levels')
axes[1].legend(loc='upper left', fontsize=9)
axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.show()

## What we've shown

1. **Lock-in is a variational quantity** (§3). The loop area in
   $(s^*, \sigma(C))$ space is exactly the cost paid in the asymmetric
   KL term that cannot be recovered after env reversal. At $\lambda = 0$
   the area is zero, as Prop 1's idealised limit requires.

2. **The cognitive-bias signature is unimodal in $\lambda$** (§4). For
   finite shock duration the loop area peaks at an intermediate $\lambda^*$
   (~5 with our parameters) and falls off in both directions: $\lambda$
   too low and there's nothing to resist; $\lambda$ too high and the
   agent can't move enough to enclose any area. The visible cost-of-
   mind-change is maximised at "stubborn but not frozen". Prop 1's
   monotonicity claim describes the *lower envelope* of this curve as
   the leg duration grows.

3. **Within $\lambda$, the residual is an equilibrium lag** (§5). Per-
   cycle areas are stable across 4 repeated shocks — the agent isn't
   accumulating damage; it's settling into a $\lambda$-modulated
   oscillation that exactly recovers each cycle's structure.

4. **Group conformity is a separate variational cost that adds to $\lambda$**
   (§6). At $\lambda = 0$ and $d = 0.9$, the loop area is large purely
   from social recursion — the network itself has a memory. Increasing
   $d$ shifts the peak $\lambda^*$ leftward: in tightly-coupled groups
   you need less individual stubbornness to produce the same observable
   hysteresis. Individual and group costs of mind-change substitute for
   each other along a ridge in $(\lambda, d)$ space.

Together these four results operationalise §3.2 of the paper:
*the complexity term is a motivational cost of mind-change, its magnitude
is directly observable as the area of the hysteresis loop, and the same
quantity can be paid by individual stubbornness, social conformity, or
any mixture of the two.*